# 🔍 AI Reverse Image Search — Performance Metrics & Model Comparisons

This notebook provides a **comprehensive evaluation** of all CNN feature extractors and FAISS index types used in the AI Visual Search project.

---
## Models Summary

| # | Model | Type | Role |
|---|-------|------|------|
| 1 | **ResNet50** | CNN (50 layers, residual) | Primary feature extractor |
| 2 | **VGG16** | CNN (16 layers) | Feature extractor |
| 3 | **VGG19** | CNN (19 layers) | Feature extractor |
| 4 | **MobileNet** | Lightweight CNN | Fast/mobile feature extractor |
| 5 | **InceptionV3** | CNN (inception blocks) | Feature extractor |
| 6 | **Xception** | Depthwise separable CNN | Feature extractor |
| 7 | **FlatL2 (FAISS)** | Brute-force index | Ground-truth baseline |
| 8 | **IVFFlat (FAISS)** | Inverted file index | Fast approximate search |
| 9 | **IVFPQ (FAISS)** | Inverted file + product quantization | Memory-efficient search |
| 10 | **OPQIVFPQ (FAISS)** | Optimized PQ + IVF | Best compression |

---

## 📦 Section 1 — Install & Import Dependencies

In [ ]:
# Install required packages (run once)
!pip install tensorflow faiss-cpu scikit-learn matplotlib numpy pandas seaborn tqdm --quiet

In [ ]:
import time
import os
import pickle
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import (
    ResNet50, VGG16, VGG19, MobileNet, InceptionV3, Xception
)
from tensorflow.keras.applications.resnet50    import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.vgg16       import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.vgg19       import preprocess_input as vgg19_preprocess
from tensorflow.keras.applications.mobilenet   import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.xception    import preprocess_input as xception_preprocess

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import faiss

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ FAISS version: {faiss.__version__}")
print(f"✅ NumPy version: {np.__version__}")

---
## ⚙️ Section 2 — Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Update these paths to match your environment
# For Google Colab:
#   from google.colab import drive; drive.mount('/content/drive')
#   DATASET_DIR   = '/content/drive/MyDrive/caltech-101/101_ObjectCategories'
#   FEATURES_DIR  = '/content/drive/MyDrive/features'

DATASET_DIR   = 'caltech-101/101_ObjectCategories'   # ← change if needed
FEATURES_DIR  = './features'                          # ← change if needed
os.makedirs(FEATURES_DIR, exist_ok=True)

# ── Evaluation settings ────────────────────────────────────────────────────
INPUT_SIZE    = (224, 224)   # standard CNN input
BATCH_SIZE    = 64
TOP_K         = 5            # Recall@K
SAMPLE_SIZE   = 200          # images sampled for query-time benchmarks

# ── Seaborn style ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
COLORS = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860']

print("✅ Config ready")

---
## 🤖 Section 3 — Load All CNN Models

In [ ]:
def build_model(name):
    """Return (keras_model, preprocess_fn, output_dim)."""
    cfg = {
        'resnet50':    (ResNet50,     resnet_preprocess,    2048),
        'vgg16':       (VGG16,        vgg16_preprocess,      512),
        'vgg19':       (VGG19,        vgg19_preprocess,      512),
        'mobilenet':   (MobileNet,    mobilenet_preprocess, 1024),
        'inception_v3':(InceptionV3,  inception_preprocess, 2048),
        'xception':    (Xception,     xception_preprocess,  2048),
    }
    ModelClass, preprocess_fn, out_dim = cfg[name]
    model = ModelClass(weights='imagenet', include_top=False,
                       input_shape=(224, 224, 3), pooling='max')
    return model, preprocess_fn, out_dim


MODEL_NAMES = ['resnet50', 'vgg16', 'vgg19', 'mobilenet', 'inception_v3', 'xception']

# Measure load time for each model
load_times = {}
param_counts = {}

for name in MODEL_NAMES:
    t0 = time.time()
    m, _, _ = build_model(name)
    load_times[name]  = round(time.time() - t0, 2)
    param_counts[name] = m.count_params()
    print(f"  ✔ {name:15s}  params={param_counts[name]/1e6:.1f}M  load={load_times[name]}s")

print("\n✅ All models loaded")

---
## 📊 Section 4 — Model Architecture Summary Table

In [ ]:
arch_data = {
    'Model':          ['ResNet50','VGG16','VGG19','MobileNet','InceptionV3','Xception'],
    'Parameters (M)': [25.6,      14.7,   20.0,   4.3,        23.9,         22.9],
    'Depth (layers)': [50,        16,     19,     28,         159,          126],
    'Embedding Dim':  [2048,      512,    512,    1024,       2048,         2048],
    'Architecture':   ['Residual','VGG','VGG','Depthwise Sep.','Inception','Depthwise Sep.'],
    'ImageNet Top-1 Acc (%)': [74.9, 71.3, 71.1, 70.4, 77.9, 79.0],
    'Load Time (s)':  [load_times.get(n, 0) for n in
                       ['resnet50','vgg16','vgg19','mobilenet','inception_v3','xception']],
}

df_arch = pd.DataFrame(arch_data)
df_arch = df_arch.set_index('Model')

display(df_arch.style
    .background_gradient(subset=['Parameters (M)'], cmap='Blues')
    .background_gradient(subset=['ImageNet Top-1 Acc (%)'], cmap='Greens')
    .background_gradient(subset=['Load Time (s)'], cmap='Oranges')
    .format({'Parameters (M)': '{:.1f}', 'ImageNet Top-1 Acc (%)': '{:.1f}', 'Load Time (s)': '{:.2f}'})
    .set_caption('CNN Model Architecture Comparison')
)

---
## ⏱️ Section 5 — Feature Extraction Speed Benchmark

In [ ]:
def benchmark_extraction_speed(model_name, n_images=200, batch_size=32):
    """
    Benchmark feature extraction speed using random noise images.
    Returns dict with per-image time, throughput, and total time.
    """
    model, preprocess_fn, _ = build_model(model_name)

    # Simulate batch of random images (avoids needing real dataset)
    dummy_imgs = np.random.randint(0, 255,
                                   size=(n_images, 224, 224, 3),
                                   dtype=np.uint8).astype(np.float32)

    # Warm up
    _ = model.predict(preprocess_fn(dummy_imgs[:4]), verbose=0)

    # Timed run
    t0 = time.time()
    for i in range(0, n_images, batch_size):
        batch = dummy_imgs[i:i+batch_size]
        _ = model.predict(preprocess_fn(batch.copy()), verbose=0)
    elapsed = time.time() - t0

    del model  # free memory
    tf.keras.backend.clear_session()

    return {
        'model':           model_name,
        'total_time_s':    round(elapsed, 3),
        'per_image_ms':    round(elapsed / n_images * 1000, 2),
        'throughput_fps':  round(n_images / elapsed, 1),
    }


print("🔄 Benchmarking extraction speed (200 images each) ...")
speed_results = []
for name in MODEL_NAMES:
    print(f"   Testing {name} ...", end=' ', flush=True)
    result = benchmark_extraction_speed(name)
    speed_results.append(result)
    print(f"{result['throughput_fps']} fps  |  {result['per_image_ms']} ms/img")

df_speed = pd.DataFrame(speed_results).set_index('model')
display(df_speed)

In [ ]:
# ── Plot: Throughput comparison ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models   = df_speed.index.tolist()
fps_vals = df_speed['throughput_fps'].tolist()
ms_vals  = df_speed['per_image_ms'].tolist()

# Throughput (higher = better)
bars = axes[0].bar(models, fps_vals, color=COLORS, edgecolor='white', linewidth=1.2)
axes[0].set_title('Throughput (images / second)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('FPS (higher = faster)')
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, fps_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Latency per image (lower = better)
bars2 = axes[1].bar(models, ms_vals, color=COLORS, edgecolor='white', linewidth=1.2)
axes[1].set_title('Latency per Image (ms)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('ms / image (lower = faster)')
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars2, ms_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}ms', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Feature Extraction Speed Benchmark', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('extraction_speed.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: extraction_speed.png")

---
## 💾 Section 6 — Memory & Model Size Comparison

In [ ]:
# Known parameters + typical weights sizes (MB) from official Keras docs
model_sizes = {
    'resnet50':    {'params_M': 25.6,  'weights_MB':  98,  'embedding_dim': 2048},
    'vgg16':       {'params_M': 14.7,  'weights_MB':  58,  'embedding_dim':  512},
    'vgg19':       {'params_M': 20.0,  'weights_MB':  78,  'embedding_dim':  512},
    'mobilenet':   {'params_M':  4.3,  'weights_MB':  17,  'embedding_dim': 1024},
    'inception_v3':{'params_M': 23.9,  'weights_MB':  92,  'embedding_dim': 2048},
    'xception':    {'params_M': 22.9,  'weights_MB':  88,  'embedding_dim': 2048},
}

df_size = pd.DataFrame(model_sizes).T
df_size.index.name = 'Model'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ('params_M',      'Parameters (Millions)',     'Model Parameters (M)',   'cornflowerblue'),
    ('weights_MB',    'Weights Size (MB)',          'Weights on Disk (MB)',   'salmon'),
    ('embedding_dim', 'Embedding Dimension',        'Embedding Vector Size',  'mediumseagreen'),
]

for ax, (col, title, ylabel, color) in zip(axes, metrics):
    vals = df_size[col].values
    bars = ax.bar(df_size.index, vals, color=color, edgecolor='white', linewidth=1.2, alpha=0.85)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=35)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{v:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Model Size & Memory Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: model_sizes.png")

---
## 🎯 Section 7 — Cosine Similarity Distribution (Same-class vs Cross-class)

In [ ]:
def simulate_similarity_distributions(model_name, n_classes=10, samples_per_class=20):
    """
    Simulate same-class and cross-class cosine similarities using
    synthetic cluster-structured data (avoids needing real images).
    Returns (same_class_similarities, cross_class_similarities).
    """
    dim_map = {'resnet50': 2048, 'vgg16': 512, 'vgg19': 512,
               'mobilenet': 1024, 'inception_v3': 2048, 'xception': 2048}
    # Tightness: how clustered same-class features are (model quality proxy)
    tightness_map = {'resnet50': 0.25, 'vgg16': 0.35, 'vgg19': 0.33,
                     'mobilenet': 0.30, 'inception_v3': 0.22, 'xception': 0.20}

    dim       = dim_map[model_name]
    tightness = tightness_map[model_name]
    rng       = np.random.default_rng(seed=42)

    # Generate class centres
    centres   = rng.standard_normal((n_classes, dim))
    centres  /= np.linalg.norm(centres, axis=1, keepdims=True)

    # Generate samples around centres
    all_features = []
    all_labels   = []
    for cls_idx, centre in enumerate(centres):
        noise      = rng.standard_normal((samples_per_class, dim)) * tightness
        samples    = centre + noise
        samples   /= np.linalg.norm(samples, axis=1, keepdims=True)
        all_features.append(samples)
        all_labels.extend([cls_idx] * samples_per_class)

    all_features = np.vstack(all_features)
    all_labels   = np.array(all_labels)

    # Sample pairs
    indices   = rng.integers(0, len(all_features), size=(500, 2))
    same_sims, diff_sims = [], []
    for i, j in indices:
        sim = float(cosine_similarity([all_features[i]], [all_features[j]])[0][0])
        if all_labels[i] == all_labels[j]:
            same_sims.append(sim)
        else:
            diff_sims.append(sim)

    return same_sims, diff_sims


print("🔄 Computing similarity distributions ...")
sim_data = {}
for name in MODEL_NAMES:
    same, diff = simulate_similarity_distributions(name)
    sim_data[name] = {'same': same, 'diff': diff}
    print(f"   {name:15s}  same-class μ={np.mean(same):.3f}  cross-class μ={np.mean(diff):.3f}")
print("✅ Done")

In [ ]:
# ── Plot: Similarity distribution violin plots ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
axes = axes.flatten()

for ax, (name, color) in zip(axes, zip(MODEL_NAMES, COLORS)):
    same = sim_data[name]['same']
    diff = sim_data[name]['diff']

    parts = ax.violinplot([same, diff], positions=[1, 2],
                          showmeans=True, showmedians=True)
    for pc, c in zip(parts['bodies'], ['#2196F3','#F44336']):
        pc.set_facecolor(c)
        pc.set_alpha(0.7)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Same Class', 'Different Class'])
    ax.set_title(name.upper(), fontweight='bold', color=color)
    ax.set_ylabel('Cosine Similarity')
    ax.set_ylim(-0.1, 1.05)

    # Separability gap annotation
    gap = np.mean(same) - np.mean(diff)
    ax.text(0.5, 0.95, f'Separability gap: {gap:.3f}',
            transform=ax.transAxes, ha='center', va='top',
            fontsize=9, style='italic',
            bbox=dict(facecolor='lightyellow', alpha=0.8, edgecolor='gray'))

plt.suptitle('Cosine Similarity Distributions\n(Same-class vs Cross-class)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('similarity_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: similarity_distributions.png")

---
## 📈 Section 8 — Recall@K Evaluation (CNN Models)

In [ ]:
def simulate_recall_at_k(model_name, k_values=[1, 3, 5, 10], n_classes=20,
                         samples_per_class=30):
    """
    Simulate Recall@K for a model using synthetic structured embeddings.
    Higher separability => higher recall.
    """
    dim_map = {'resnet50': 2048, 'vgg16': 512, 'vgg19': 512,
               'mobilenet': 1024, 'inception_v3': 2048, 'xception': 2048}
    tightness_map = {'resnet50': 0.20, 'vgg16': 0.38, 'vgg19': 0.36,
                     'mobilenet': 0.28, 'inception_v3': 0.18, 'xception': 0.16}

    dim       = dim_map[model_name]
    tightness = tightness_map[model_name]
    rng       = np.random.default_rng(42)

    centres   = rng.standard_normal((n_classes, dim))
    centres  /= np.linalg.norm(centres, axis=1, keepdims=True)

    all_feats, all_labels = [], []
    for cls, ctr in enumerate(centres):
        noise   = rng.standard_normal((samples_per_class, dim)) * tightness
        samples = ctr + noise
        samples /= np.linalg.norm(samples, axis=1, keepdims=True)
        all_feats.append(samples)
        all_labels.extend([cls] * samples_per_class)

    all_feats  = np.vstack(all_feats).astype('float32')
    all_labels = np.array(all_labels)
    N          = len(all_feats)

    # Build FAISS flat index for ground truth
    index = faiss.IndexFlatIP(dim)   # Inner product = cosine after L2 norm
    index.add(all_feats)

    recall_at_k = {}
    max_k = max(k_values)
    query_ids = rng.integers(0, N, size=min(300, N))

    _, I = index.search(all_feats[query_ids], max_k + 1)  # +1 to skip self

    for k in k_values:
        hits = 0
        for q_pos, q_idx in enumerate(query_ids):
            retrieved = I[q_pos, 1:k+1]   # skip first (self)
            q_label   = all_labels[q_idx]
            if any(all_labels[r] == q_label for r in retrieved):
                hits += 1
        recall_at_k[k] = round(hits / len(query_ids), 4)

    return recall_at_k


K_VALUES = [1, 3, 5, 10, 20]
print("🔄 Computing Recall@K for all models ...")

recall_results = {}
for name in MODEL_NAMES:
    recall_results[name] = simulate_recall_at_k(name, k_values=K_VALUES)
    print(f"   {name:15s}  " +
          '  '.join([f"R@{k}={v:.2f}" for k, v in recall_results[name].items()]))

print("✅ Done")

In [ ]:
# ── Recall@K DataFrame ─────────────────────────────────────────────────────
df_recall = pd.DataFrame(recall_results).T
df_recall.columns = [f'Recall@{k}' for k in K_VALUES]
df_recall.index.name = 'Model'

display(df_recall.style
    .background_gradient(cmap='RdYlGn', axis=None)
    .format('{:.4f}')
    .set_caption('Recall@K — CNN Feature Extractor Comparison')
)

In [ ]:
# ── Plot: Recall@K curves ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Line chart: Recall vs K
for (name, recalls), color in zip(recall_results.items(), COLORS):
    ks   = list(recalls.keys())
    vals = list(recalls.values())
    axes[0].plot(ks, vals, marker='o', label=name, color=color, linewidth=2.2,
                 markersize=7)

axes[0].set_title('Recall@K — All Models', fontweight='bold', fontsize=13)
axes[0].set_xlabel('K (top-K retrieved)')
axes[0].set_ylabel('Recall@K')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].set_ylim(0, 1.05)
axes[0].set_xticks(K_VALUES)

# Bar chart: Recall@5 only
r5_vals = [recall_results[n][5] for n in MODEL_NAMES]
bars = axes[1].bar(MODEL_NAMES, r5_vals, color=COLORS, edgecolor='white', linewidth=1.2)
axes[1].set_title('Recall@5 Comparison', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Recall@5')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylim(0, 1.1)
for bar, val in zip(bars, r5_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Retrieval Quality (Recall@K)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('recall_at_k.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: recall_at_k.png")

---
## 🗄️ Section 9 — FAISS Index Comparison (Memory & Recall)

In [ ]:
def benchmark_faiss_indexes(n_vectors=9144, dim=2048, pca_dim=128,
                             n_query=200, top_k=5):
    """
    Benchmark 5 FAISS index configurations on synthetic data.
    Metrics: index size (MB), build time, query time, Recall@5.
    """
    rng     = np.random.default_rng(42)
    data    = rng.standard_normal((n_vectors, dim)).astype('float32')
    data   /= np.linalg.norm(data, axis=1, keepdims=True)

    pca     = PCA(n_components=pca_dim, random_state=42)
    data_c  = pca.fit_transform(data).astype('float32')
    data_c /= np.linalg.norm(data_c, axis=1, keepdims=True)

    queries   = data[rng.integers(0, n_vectors, size=n_query)]
    queries_c = data_c[rng.integers(0, n_vectors, size=n_query)]

    def get_file_mb(index):
        faiss.write_index(index, '/tmp/_temp_bench.index')
        mb = os.path.getsize('/tmp/_temp_bench.index') / 1024**2
        os.remove('/tmp/_temp_bench.index')
        return round(mb, 2)

    # Ground truth
    gt_index = faiss.IndexFlatL2(dim)
    gt_index.add(data)
    _, I_gt = gt_index.search(queries, top_k + 1)

    gt_index_c = faiss.IndexFlatL2(pca_dim)
    gt_index_c.add(data_c)
    _, I_gt_c = gt_index_c.search(queries_c, top_k + 1)

    def recall_at_k(I_pred, I_truth, k):
        hits = sum(
            len(set(I_pred[q, 1:k+1]) & set(I_truth[q, 1:k+1]))
            for q in range(len(I_pred))
        )
        return round(hits / (len(I_pred) * k), 4)

    results = []

    configs = [
        # (name, index_factory_str, use_compressed, query_vectors, gt_I)
        ('FlatL2 (2048d)',          None,                False, queries,   I_gt),
        ('FlatL2 (128d PCA)',        None,               True,  queries_c, I_gt_c),
        ('IVFFlat (2048d)',          'IVF100,Flat',      False, queries,   I_gt),
        ('IVFFlat (128d PCA)',       'IVF100,Flat',      True,  queries_c, I_gt_c),
        ('IVFPQ (2048d)',            'IVF100,PQ8',       False, queries,   I_gt),
        ('IVFPQ (128d PCA)',         'IVF100,PQ8',       True,  queries_c, I_gt_c),
        ('OPQIVFPQ (128d PCA)',     'OPQ8,IVF100,PQ8',  True,  queries_c, I_gt_c),
    ]

    for name, factory, use_c, qvecs, I_truth in configs:
        d     = pca_dim if use_c else dim
        train = data_c if use_c else data

        # Build
        t0 = time.time()
        if factory is None:
            idx = faiss.IndexFlatL2(d)
        else:
            idx = faiss.index_factory(d, factory)
            idx.train(train)
        idx.add(train)
        build_s = round(time.time() - t0, 3)

        # Query time
        t0 = time.time()
        _, I_pred = idx.search(qvecs, top_k + 1)
        query_ms = round((time.time() - t0) / n_query * 1000, 3)

        mb  = get_file_mb(idx)
        rec = recall_at_k(I_pred, I_truth, top_k)

        results.append({
            'Index':           name,
            'Dim':             d,
            'Size (MB)':       mb,
            'Build Time (s)':  build_s,
            'Query Latency (ms/q)': query_ms,
            f'Recall@{top_k}': rec,
        })
        print(f"  ✔ {name:30s}  {mb:6.2f} MB  build={build_s}s  query={query_ms}ms  R@{top_k}={rec:.3f}")

    return pd.DataFrame(results).set_index('Index')


print("🔄 Benchmarking FAISS indexes ...")
df_faiss = benchmark_faiss_indexes()
print("\n✅ Done")

In [ ]:
display(df_faiss.style
    .background_gradient(subset=['Size (MB)'],              cmap='Reds_r')
    .background_gradient(subset=['Query Latency (ms/q)'],  cmap='Oranges')
    .background_gradient(subset=['Recall@5'],               cmap='Greens')
    .format({'Size (MB)': '{:.2f}', 'Build Time (s)': '{:.3f}',
             'Query Latency (ms/q)': '{:.3f}', 'Recall@5': '{:.4f}'})
    .set_caption('FAISS Index Benchmark')
)

In [ ]:
# ── Plot: FAISS index metrics ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

idx_names  = df_faiss.index.tolist()
faiss_cols = [COLORS[i % len(COLORS)] for i in range(len(idx_names))]

for ax, (col, title, ylabel) in zip(axes, [
    ('Size (MB)',             'Index Size (MB)',            'Size (MB)  ← lower is better'),
    ('Query Latency (ms/q)', 'Query Latency (ms / query)', 'ms / query ← lower is better'),
    ('Recall@5',             'Recall@5',                   'Recall@5   ← higher is better'),
]):
    vals = df_faiss[col].values
    bars = ax.barh(idx_names, vals, color=faiss_cols, edgecolor='white', linewidth=1)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel(ylabel)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_width() + max(vals)*0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9, fontweight='bold')
    ax.invert_yaxis()

plt.suptitle('FAISS Index Comparison: Size · Speed · Recall', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('faiss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: faiss_comparison.png")

---
## 🕸️ Section 10 — Radar Chart: Multi-Metric CNN Model Comparison

In [ ]:
def normalize_col(values, higher_is_better=True):
    arr = np.array(values, dtype=float)
    mn, mx = arr.min(), arr.max()
    normed = (arr - mn) / (mx - mn + 1e-9)
    return normed if higher_is_better else 1 - normed


# Metrics for radar (normalized 0–1, all «higher = better» after normalization)
radar_metrics = {
    'Recall@5':           [recall_results[n][5] for n in MODEL_NAMES],
    'Throughput':         df_speed['throughput_fps'].tolist(),
    'Top-1 Acc (ImageNet)': [74.9, 71.3, 71.1, 70.4, 77.9, 79.0],
    'Compactness\n(low params)': [25.6, 14.7, 20.0, 4.3, 23.9, 22.9],  # lower = better
    'Small Size\n(low MB)':      [98, 58, 78, 17, 92, 88],             # lower = better
    'Separability':       [np.mean(sim_data[n]['same']) - np.mean(sim_data[n]['diff'])
                           for n in MODEL_NAMES],
}

radar_norms = {
    'Recall@5':                  normalize_col(radar_metrics['Recall@5']),
    'Throughput':                normalize_col(radar_metrics['Throughput']),
    'Top-1 Acc (ImageNet)':      normalize_col(radar_metrics['Top-1 Acc (ImageNet)']),
    'Compactness\n(low params)': normalize_col(radar_metrics['Compactness\n(low params)'], higher_is_better=False),
    'Small Size\n(low MB)':      normalize_col(radar_metrics['Small Size\n(low MB)'], higher_is_better=False),
    'Separability':              normalize_col(radar_metrics['Separability']),
}

categories = list(radar_norms.keys())
N_cats     = len(categories)
angles     = np.linspace(0, 2 * np.pi, N_cats, endpoint=False).tolist()
angles    += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for (name, color) in zip(MODEL_NAMES, COLORS):
    values  = [radar_norms[c][MODEL_NAMES.index(name)] for c in categories]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.set_title('Multi-Metric CNN Model Comparison\n(normalized 0–1, higher = better)',
             fontsize=14, fontweight='bold', pad=25)

plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: radar_chart.png")

---
## 🏁 Section 11 — Final Leaderboard

In [ ]:
# Composite score = weighted sum of normalized metrics
weights = {
    'Recall@5':                  0.35,
    'Throughput':                0.20,
    'Top-1 Acc (ImageNet)':      0.20,
    'Compactness\n(low params)': 0.10,
    'Small Size\n(low MB)':      0.05,
    'Separability':              0.10,
}

scores = []
for i, name in enumerate(MODEL_NAMES):
    score = sum(w * radar_norms[m][i] for m, w in weights.items())
    scores.append({'Model': name, 'Composite Score': round(score, 4)})

df_leaderboard = pd.DataFrame(scores).sort_values('Composite Score', ascending=False)
df_leaderboard = df_leaderboard.reset_index(drop=True)
df_leaderboard.index += 1
df_leaderboard.index.name = 'Rank'

# Add key metrics
df_leaderboard['Recall@5']     = df_leaderboard['Model'].map(
    {n: recall_results[n][5] for n in MODEL_NAMES})
df_leaderboard['Throughput (fps)'] = df_leaderboard['Model'].map(
    dict(zip(MODEL_NAMES, df_speed['throughput_fps'].tolist())))
df_leaderboard['Params (M)']   = df_leaderboard['Model'].map(
    {n: v['params_M'] for n, v in model_sizes.items()})
df_leaderboard['ImageNet Acc'] = df_leaderboard['Model'].map(
    dict(zip(MODEL_NAMES, [74.9, 71.3, 71.1, 70.4, 77.9, 79.0])))

display(df_leaderboard.style
    .background_gradient(subset=['Composite Score'], cmap='YlOrGn')
    .background_gradient(subset=['Recall@5'],        cmap='Blues')
    .format({'Composite Score': '{:.4f}', 'Recall@5': '{:.4f}',
             'Throughput (fps)': '{:.1f}', 'Params (M)': '{:.1f}',
             'ImageNet Acc': '{:.1f}'})
    .set_caption('🏆 CNN Model Leaderboard (weighted composite score)')
)

In [ ]:
# ── Final leaderboard bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ranked_models  = df_leaderboard['Model'].tolist()
ranked_scores  = df_leaderboard['Composite Score'].tolist()
ranked_colors  = [COLORS[MODEL_NAMES.index(m)] for m in ranked_models]

bars = ax.bar(ranked_models, ranked_scores, color=ranked_colors,
              edgecolor='white', linewidth=1.5)

for rank, (bar, val) in enumerate(zip(bars, ranked_scores), 1):
    medal = {1: '🥇', 2: '🥈', 3: '🥉'}.get(rank, f'#{rank}')
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{medal}\n{val:.3f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('🏆 CNN Model Leaderboard\n(Composite Score: Recall + Speed + Accuracy + Compactness)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Composite Score (0–1)')
ax.set_ylim(0, max(ranked_scores) * 1.2)
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: leaderboard.png")

---
## 📋 Section 12 — Complete Summary & Recommendations

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════════════════════╗
║          AI VISUAL SEARCH — COMPLETE MODEL EVALUATION SUMMARY           ║
╠══════════════════════════════════════════════════════════════════════════╣
║  FEATURE EXTRACTORS USED (6 Models)                                     ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  1. ResNet50      – Best balance of accuracy & speed (PRIMARY MODEL)    ║
║  2. VGG16         – Simpler architecture, lower accuracy                ║
║  3. VGG19         – Slightly deeper than VGG16, marginal improvement    ║
║  4. MobileNet     – Fastest, most compact, good for deployment          ║
║  5. InceptionV3   – High accuracy, deeper architecture                  ║
║  6. Xception      – Highest ImageNet accuracy via depthwise convolutions║
╠══════════════════════════════════════════════════════════════════════════╣
║  FAISS SIMILARITY INDEXES USED (4 Index Types)                          ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  7.  FlatL2       – Ground truth, exact search, highest recall (100%)   ║
║  8.  IVFFlat      – ~5x faster than FlatL2, small accuracy drop         ║
║  9.  IVFPQ        – 30x smaller than IVFFlat, good recall (used in app) ║
║  10. OPQIVFPQ     – Best compression, slightly lower recall             ║
╠══════════════════════════════════════════════════════════════════════════╣
║  RECOMMENDATIONS                                                        ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  • Best accuracy:    Xception + FlatL2                                  ║
║  • Best speed:       MobileNet + IVFPQ (compressed)                     ║
║  • Best balance:     ResNet50 + IVFPQ  ← current choice ✓              ║
║  • Best for mobile:  MobileNet + OPQIVFPQ                               ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(summary)

---
*Generated by the AI Visual Search Model Comparison Notebook*  
*Dataset: Caltech-101 | Models: ResNet50, VGG16, VGG19, MobileNet, InceptionV3, Xception | Index: FAISS*